### Data input maker

In [2]:
import numpy as np
import pandas as pd
import pickle
import dill
import networkx as nx
from pathlib import Path
from scipy.sparse import load_npz, save_npz
from tqdm.auto import tqdm
import shutil

In [3]:
# Working directory
BASE_DIR = Path('/Users/gre_en/Documents/Analysis/Projects/1_research/Sci-Soc')

# Processed data directory
NETWORKS_DIR = BASE_DIR / "data" / "processed" / "networks"    # Network adj matrix
EMB_DIR = BASE_DIR / "data" / "processed" / "embeddings"       # Embedding vector directory
DYSAT_DIR = BASE_DIR / "data" / "processed" / "dysat_input"    # Input for the official DySAT code

# Result directory
PLOT_DIR = BASE_DIR / "results" / "figures"

DYSAT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 1990
YEAR_END   = 2023
years = list(range(YEAR_START, YEAR_END + 1))

RANDOM_SEED = 42
SOURCES = ["news", "paper"]

In [17]:
def load_node_info(source, net_dir):
    with open(Path(net_dir) / f"{source}_node_info.pkl", "rb") as f:
        return pickle.load(f)

info = {"news": load_node_info("news", NETWORKS_DIR),
        "paper": load_node_info("paper", NETWORKS_DIR)}

# sanity check
print(len(list(NETWORKS_DIR.glob("*_fixed_adj_*.npz"))), "matrices")
print(info["news"].keys())
print("news  years", len(info["news"]["years"]), "V_t", info["news"]["V_t"][0], "->", info["news"]["V_t"][-1])
print("paper years", len(info["paper"]["years"]), "V_t", info["paper"]["V_t"][0], "->", info["paper"]["V_t"][-1])

68 matrices
dict_keys(['vocab', 'years', 'concept_freq_year', 'concept_freq_total', 'concept_share', 'concept_share_all', 'n_docs_all', 'n_docs_ge2', 'V_t', 'E_t'])
news  years 34 V_t 2434 -> 7883
paper years 34 V_t 12593 -> 18159


In [ ]:
def to_dysat_input(source, info, net_dir, out_dir, min_docs=1):
    out_dir = Path(out_dir) / source
    out_dir.mkdir(parents=True, exist_ok=True)

    keep = np.where(np.asarray(info["concept_freq_total"]) >= min_docs)[0]
    for year in info["years"]:
        W = load_npz(Path(net_dir) / f"{source}_fixed_adj_{year}.npz").tocsr()
        save_npz(out_dir / f"adj_{year}.npz", W[keep][:, keep])

    np.save(out_dir / "years.npy", np.asarray(info["years"]))
    np.save(out_dir / "keep.npy", keep)                    # local -> global
    np.save(out_dir / "vocab.npy",
            np.array(info["vocab"], dtype=object)[keep])
    np.save(out_dir / "active.npy",
            np.asarray(info["concept_freq_year"])[:, keep] > 0)
    return len(keep)


for s in SOURCES:
    n = to_dysat_input(s, info[s], NETWORKS_DIR, DYSAT_DIR)
    print(s, "nodes", n)

news nodes 14353


In [ ]:
for s in SOURCES:
    d = DYSAT_DIR / s
    files = sorted(d.glob("adj_*.npz"))
    print(s, len(files), "files",
          f"{sum(f.stat().st_size for f in d.iterdir())/1e6:.1f} MB")
    if files:
        print("  ", files[-1].name, f"{files[-1].stat().st_size/1e6:.2f} MB")

In [ ]:
for s in SOURCES:
    f = info[s]["concept_freq_total"]
    n = len(f)
    print(s, "전체", n,
          "| 한번도 안나옴", (f == 0).sum(),
          "| df==1", (f == 1).sum(),
          "| df>=5", (f >= 5).sum())

### Test

In [6]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from pathlib import Path
from scipy.sparse import load_npz
from tqdm.auto import tqdm


def normalize_gcn(adj):
    """D^-1/2 (A+I) D^-1/2, computed per edge so nnz stays fixed."""
    A = sp.coo_matrix(adj, dtype=np.float32) + sp.eye(adj.shape[0], dtype=np.float32)
    A = A.tocoo()
    d = np.asarray(A.sum(1)).flatten()
    dinv = np.zeros_like(d)
    nz = d > 0
    dinv[nz] = 1.0 / np.sqrt(d[nz])
    data = A.data * dinv[A.row] * dinv[A.col]
    return sp.coo_matrix((data, (A.row, A.col)), shape=A.shape)



In [7]:
from scipy.sparse import load_npz
import numpy as np

d = DYSAT_DIR / "news"
years = np.load(d / "years.npy")
for y in [years[0], years[len(years)//2], years[-1]]:
    W = load_npz(d / f"adj_{y}.npz")
    A = normalize_gcn(W)
    print(y, "shape", W.shape, "nnz", W.nnz, "-> normalized nnz", A.nnz)

1990 shape (14353, 14353) nnz 87224 -> normalized nnz 101577
2007 shape (14353, 14353) nnz 485602 -> normalized nnz 499955
2023 shape (14353, 14353) nnz 889070 -> normalized nnz 903423
